In [4]:
import sys, os
sys.path.insert(0, os.path.abspath("/notebooks/"))

import torch
from pathlib import Path
from src.tokenizer import INSTR_OFF, INSTR_NAMES, BOS, EOS, BAR, tokens_to_score
from src.model import load_model, CHECKPOINT_DIR
from src.generator import show_text_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
INFILL_WEIGHTS_FILE = CHECKPOINT_DIR / "infill_model.pt"

print("Available instruments:")
for i, name in enumerate(INSTR_NAMES):
    print(f"  {i:2d}: {name}")

Available instruments:
   0: other
   1: piano
   2: harpsichord
   3: organ
   4: accordion
   5: guitar
   6: bass_guitar
   7: violin
   8: viola
   9: cello
  10: double_bass
  11: harp
  12: flute
  13: oboe
  14: clarinet
  15: bassoon
  16: saxophone
  17: horn
  18: trumpet
  19: trombone
  20: tuba
  21: percussion
  22: timpani
  23: celesta
  24: voice_soprano
  25: voice_mezzo
  26: voice_alto
  27: voice_tenor
  28: voice_baritone
  29: voice_bass
  30: voice
  31: choir


### Generation Parameters

In [12]:
# Instruments to include in the generated score (names from INSTR_NAMES above)
INSTRUMENTS = ["voice"]

# Number of measures to generate
N_MEASURES = 2

# Sampling parameters
TEMPERATURE = 0.4
TOP_K       = 50
MAX_NEW     = 1500

### Generate

In [13]:
# Resolve instrument indices and build header tokens
instr_indices = []
for name in INSTRUMENTS:
    name_lower = name.lower()
    if name_lower in INSTR_NAMES:
        instr_indices.append(INSTR_NAMES.index(name_lower))
    else:
        print(f"Unknown instrument '{name}', falling back to 'other'")
        instr_indices.append(0)

header_tokens = [INSTR_OFF + idx for idx in instr_indices]
print(f"Instruments: {INSTRUMENTS}")
print(f"Header tokens: {header_tokens}")

# Load infill model
print(f"\nLoading model from {INFILL_WEIGHTS_FILE}...")
model, cfg = load_model(DEVICE)
if INFILL_WEIGHTS_FILE.exists():
    state = torch.load(INFILL_WEIGHTS_FILE, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state)
    print("Loaded infill weights.")
else:
    print("Infill weights not found — using base model weights.")
print(f"Model loaded ({model.num_params():,} params)")

# Generate from scratch: empty prefix and suffix, the model fills the entire gap
print(f"\nGenerating {N_MEASURES} measures from scratch...")
fill_tokens = model.generate_infill(
    prefix_tokens=[],
    suffix_tokens=[],
    header_tokens=header_tokens,
    target_bars=N_MEASURES,
    max_new=MAX_NEW,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    device=DEVICE,
)

generated_bars = fill_tokens.count(BAR)
print(f"Generated {len(fill_tokens)} tokens ({generated_bars} bars)")

Instruments: ['voice']
Header tokens: [461]

Loading model from storage\checkpoints\infill_model.pt...
Loaded infill weights.
Model loaded (20,425,728 params)

Generating 2 measures from scratch...
Written 1 bars of 2
Written 2 bars of 2
Written 3 bars of 2
Generated 683 tokens (2 bars)


### Export

In [14]:
# Reconstruct score from generated tokens
full_tokens = [BOS] + header_tokens + fill_tokens + [EOS]
score = tokens_to_score(full_tokens)

# Color generated notes red for easy identification
for el in score.flatten().notesAndRests:
    if el.isNote or el.isChord:
        el.style.color = 'red'

# Build output filename from instruments and measure count
instr_tag = "_".join(INSTRUMENTS)
name = f"scratch_{instr_tag}_{N_MEASURES}bars"
midi_path = str(OUTPUT_DIR / f"{name}.mid")
xml_path  = str(OUTPUT_DIR / f"{name}.xml")

score.write("midi", fp=midi_path)
print(f"Saved MIDI -> {midi_path}")

try:
    score.makeNotation().write("musicxml", fp=xml_path)
    print(f"Saved MusicXML -> {xml_path}")
except Exception as e:
    print(f"MusicXML export skipped: {e}")
    xml_path = None

note_count   = sum(1 for el in score.flatten().notesAndRests if el.isNote or el.isChord)
measure_count = len(score.parts[0].getElementsByClass("Measure")) if score.parts else 0
print(f"\nScore summary: {measure_count} measures, {note_count} notes/chords")

if xml_path:
    print("\n--- Score Preview ---")
    show_text_score(xml_path, print)

Saved MIDI -> output\scratch_voice_2bars.mid
MusicXML export skipped: '1'

Score summary: 2 measures, 96 notes/chords
